# AGEC:370 Agricultural Price Analysis

## Exam 2 Study Guide:

This notebook reviews concepts and methods for Exam 2:

- Four determinants of agricultural prices
- Nominal vs. real prices and CPI deflation
- Lagged variables: what `shift(1)` does and why
- Log transformations: levels vs. log scale
- Level-level vs. log-log regression interpretation
- R-squared and model fit
- USDA NASS data codes `(D)` and `(Z)`
- AR(1) autoregressive models
- Train-test split

We will use USDA NASS national-level data on corn prices, production, and planted acres.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chuckgrigsby0/agec-370/blob/main/notebooks/24_study_guide_exam_02.ipynb)

## Data Loading

All data loads directly from GitHub.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_style('whitegrid')

base_url = 'https://raw.githubusercontent.com/chuckgrigsby0/agec-370/main/data/'

# Full price history (1866-2024) kept for the nominal vs. real visualization
corn_price_hist = pd.read_csv(base_url + 'corn_prices_bushels_nominal_real.csv')
corn_price_hist = corn_price_hist[['year', 'nominal_price', 'real_price', 'annual_avg_cpi']].sort_values('year')

# Production (billion bushels) and planted acres (million acres) for regression
corn_prod    = pd.read_csv(base_url + 'corn_production_bushels_nass.csv')[['year', 'corn_prod_bu']]
corn_planted = pd.read_csv(base_url + 'corn_planted_acres_nass.csv')[['year', 'corn_planted_acres']]

### Join production, price, and planted acreage data

In [ ]:
# Inner join keeps only years present in all three datasets
corn_dta = pd.merge(corn_prod, corn_price_hist, on='year', how='inner')
corn_dta = pd.merge(corn_dta, corn_planted, on='year', how='inner')
corn_dta = corn_dta.sort_values('year').reset_index(drop=True)

select_columns = ['year', 'corn_prod_bu', 'corn_planted_acres', 'nominal_price', 'real_price', 'annual_avg_cpi']
corn_dta = corn_dta[select_columns].copy()


min_yr, max_yr = corn_dta['year'].min(), corn_dta['year'].max()
print(f'Merged data: {min_yr}-{max_yr}  ({len(corn_dta)} observations)')
corn_dta.head()

## The Four Determinants of Agricultural Prices

Agricultural prices affected by four primary factors:

| Determinant | Time Horizon | Example |
|---|---|---|
| **Changes in Long-Run Supply and Demand** | Structural shifts over multiple years (e.g., decades) | Real corn prices falling as technology raises yields |
| **Seasonalily** | Predictable within-year patterns | Milk peaks in spring; grain prices fall at harvest |
| **Market shocks** | Unexpected, temporary disruptions | Avian influenza causing egg price spikes (2015, 2024) |
| **Market Adjustments** | How markets respond to shocks given production lags (e.g., 1-3 years) | Farmers respond to high prices by planting more *next* year |

### Why Prices Overshoot After a Shock

When a supply shock hits, prices spike immediately, but production cannot respond within the same crop year. Farmers observe high prices and expand production the **following** year, often overshooting equilibrium. Prices then fall below equilibrium, triggering another correction. 

**Key point:** Production lags (biological and planning constraints) are a primary reason agricultural prices fluctuate rather than immediately returning to equilibrium after a shock.

### Plot Nominal vs Real Corn Prices

In [ ]:
# Nominal prices rise with general inflation; real prices reveal the true long-run downward trend
price_plot = corn_price_hist.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(price_plot['year'], price_plot['nominal_price'], color='steelblue', linewidth=1.2)
axes[0].set_title('Nominal Corn Prices ($/BU)', fontsize=12, weight='bold')
axes[0].set_xlabel('Year', weight='bold')
axes[0].set_ylabel('Nominal Price ($/BU)', weight='bold')

axes[1].plot(price_plot['year'], price_plot['real_price'], color='darkred', linewidth=1.2)
axes[1].set_title('Real Corn Prices (2024 $/BU)', fontsize=12, weight='bold')
axes[1].set_xlabel('Year', weight='bold')
axes[1].set_ylabel('Real Price ($/BU)', weight='bold')

plt.suptitle('Nominal prices trend upward with inflation; real prices reveal the true downward long-run trend',
             fontsize=10, style='italic')
plt.tight_layout()
plt.show()

## Nominal vs. Real Prices

**Nominal price**: the market price in current dollars at the time of observation.

**Real price**: the nominal price adjusted for inflation, expressed in constant dollars of a base year (here, 2024).

### CPI Deflation Formula

$$\text{Real Price}_t = \frac{\text{Nominal Price}_t}{\text{CPI}_t} \times \text{CPI}_{\text{base}}$$

Where $\text{CPI}_t$ is the price level in year $t$ and $\text{CPI}_{\text{base}}$ is the price level in the base year.

### Worked Example

In 1990, nominal corn price = \$2.28/bu. $\text{CPI}_{1990} = 130.7$, $\text{CPI}_{2024} = 313.689$ (base year = 2024).

$$\text{Real Price}_{1990} = \frac{2.28}{130.7} \times 313.689 \approx \$5.47 \text{ (2024 \$/bu)}$$

The real price is **higher** than the nominal because general prices rose between 1990 and 2024 - \$2.28 in 1990 dollars has the same purchasing power as $5.47 in 2024 dollars. The nominal price in 1990 appears lower because general price levels were lower in 1990. 

In [ ]:
# Replicate the CPI deflation formula step by step
nominal_1990 = 2.28
cpi_1990     = 130.7
cpi_2024     = 313.689  # base year

real_2024 = (nominal_1990 / cpi_1990) * cpi_2024

print(f'Nominal price (1990):      ${nominal_1990:.2f}/bu')
print(f'Real price (2024 dollars): ${real_2024:.2f}/bu')
print(f'Formula: ({nominal_1990} / {cpi_1990}) x {cpi_2024} = {real_2024:.4f}')

## Lagged Variables

A **lagged variable** shifts a column back by one period. `shift(1)` moves each value *down* one row, so the value in row $t$ becomes the value from period $t-1$.

### Why use lags in regressions?

Producers make planting decisions in spring based on prices from the **previous** year. They cannot observe the market price in advance. Using `real_price_lag1` as a predictor captures this decision-making process.

**Economic logic:** Higher prices last year → expected profitability → more acres planted this year.

The output below shows exactly what `shift(1)` does row by row.

In [ ]:
# Before: each row shows the real price for that calendar year. No lag applied yet
print('Before shift(1):')
print(corn_dta[['year', 'real_price']].head(6).to_string(index=False))

In [ ]:
# After shift(1): each price moves down one row
# In the 1927 row, real_price_lag1 = the 1926 price, which is the price the producer knew at planting time
corn_dta['real_price_lag1'] = corn_dta['real_price'].shift(1)

print('After shift(1) — real_price_lag1 holds the prior-year price:')
print(corn_dta[['year', 'real_price', 'real_price_lag1']].head(6).to_string(index=False))
print('\nRow 0 is NaN: no prior-year price exists before the first observation.')

### Reading the Lagged Output

| year | real_price | real_price_lag1 |
|------|-----------|------------------|
| 1926 | 12.67     | NaN              |
| 1927 | 14.49     | **12.67**        |
| 1928 | 14.64     | **14.49**        |

In the 1927 row, `real_price_lag1 = 12.67`, which is the 1926 price. This is what a producer observed when making planting decisions for 1927. The value shifted down from the row above.

The first row is always `NaN` because there is no prior year in the dataset. Drop it with `dropna()` before running any regression.

In [ ]:
# dropna() removes the NaN row created by shift(1); reset t to start at 0
corn_dta = corn_dta.dropna().reset_index(drop=True)
corn_dta['t'] = range(len(corn_dta))  # time trend variable for regression

min_yr, max_yr = corn_dta['year'].min(), corn_dta['year'].max()
print(f'Analysis dataset: {min_yr}-{max_yr}  ({len(corn_dta)} observations)')

## Log Transformations

Taking the natural log of a variable (`np.log(x)`) changes both the scale and the interpretation of regression coefficients.

**Why log-transform a variable?**

1. **Linearizes exponential growth** - corn production grew roughly exponentially; logging it makes the trend approximately linear, which OLS handles well
2. **Reduces right skew** - pulls large values closer to the center of the distribution
3. **Enables elasticity interpretation** - the slope coefficient becomes the % change in $Y$ per 1% change in $X$

The plots below show the same corn production data in levels (left) and on a log scale (right).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(corn_dta['year'], corn_dta['corn_prod_bu'],
             color='darkblue', linewidth=1.2)
axes[0].set_title('Corn Production — Levels', fontsize=11, weight='bold')
axes[0].set_xlabel('Year', weight='bold')
axes[0].set_ylabel('Billion Bushels', weight='bold')

axes[1].plot(corn_dta['year'], np.log(corn_dta['corn_prod_bu']),
             color='darkorange', linewidth=1.2)
axes[1].set_title('ln(Corn Production) — Log Scale', fontsize=11, weight='bold')
axes[1].set_xlabel('Year', weight='bold')
axes[1].set_ylabel('ln(Billion Bushels)', weight='bold')

plt.suptitle('Logging compresses exponential growth into a roughly linear trend',
             fontsize=10, style='italic')
plt.tight_layout()
plt.show()

## OLS Regression: Level-Level Model

**Model:** $\text{Corn Acres Planted} = \beta_0 + \beta_1 \cdot \text{Real Price}_{t-1} + \beta_2 \cdot t + \varepsilon$

Both the dependent variable and the predictor are in **original units** (million acres and \$/bu).

### Coefficient Interpretation - Level-Level

> **"A \$1 increase in last year's real corn price is associated with a change of $\hat{\beta}_1$ million acres planted this year, holding the time trend constant."**

**Scaling for smaller price changes:**

- For a **\$0.10/bu change**: $\Delta \text{Acres} = \hat{\beta}_1 \times 0.10$
- For a **\$1.00/bu change**: use $\hat{\beta}_1$ directly

**Note:** $\Delta \text{Acres}$ means change in acres planted.

**Expected sign:** We expect a positive association between last year's corn prices and acres planted. Higher prices last year signal profitability, incentivizing farmers to plant more corn.

### Plot planted acres

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(corn_dta['year'], corn_dta['corn_planted_acres'],
        color='darkorange', linewidth=1.2)
ax.set_title('Corn Acres Planted', fontsize=11, weight='bold')
ax.set_xlabel('Year', weight='bold')
ax.set_ylabel('Millions of Acres', weight='bold')

plt.tight_layout()
plt.show()

Corn acreage generally declined from the 1920s through the early 1970s before recovering through the present. Capturing this nonlinear trend fully would require squared and cubic time terms. For simplicity, we restrict the data to 1960 and later, a period over which a linear trend provides a reasonable approximation.

In [ ]:
# Subset to years >= 1960. Pre-1960 data reflect different structural conditions
# Create a new time trend variable starting at 0 for the subset
corn_dta_subset = corn_dta[corn_dta['year'] >= 1960].copy()
corn_dta_subset = corn_dta_subset.sort_values('year').reset_index(drop=True)
corn_dta_subset['t'] = range(len(corn_dta_subset))

In [ ]:
corn_dta_subset.head()

In [ ]:
model1 = smf.ols('corn_planted_acres ~ real_price_lag1 + t ', data=corn_dta_subset).fit()
print(model1.summary())

### Reading the Level-Level Output

Focus on the `coef` column:

- **`Intercept`** (~49.7 million acres): predicted acreage when `real_price_lag1 = 0` and `t = 0`, the baseline level
- **`real_price_lag1`** (~1.465 million acres per \$/bu): for every **\$1** increase in last year's real corn price, acreage increases by ~1.465 million acres
  - For a **\$0.10/bu increase**: $1.465 \times 0.10 = 0.1465$ million acres (~146,500 acres)
  - Positive sign confirms the expected positive association between last year's price and this year's planting acreage
- **`t`**: controls for gradual long-run structural shifts in planted acreage
- **`P>|t|`** on `real_price_lag1` < 0.05: statistically significant once the time trend is controlled
- **`R-squared`** (~0.754): Approximately 75% of the variation of acreage planted is explained by lagged prices and trend.

## OLS Regression: Log-Log Model (Corn Acres Planted)

**Model:** $\ln(\text{Corn Acres Planted}) = \beta_0 + \beta_1 \cdot \ln(\text{Real Price}_{t-1}) + \beta_2 \cdot t + \varepsilon$

Both the dependent variable and the predictor are **log-transformed**. This is the **constant-elasticity** model.

### Coefficient Interpretation - Log-Log

> **"A 1% increase in last year's real corn price is associated with a $\hat{\beta}_1$% increase in corn planted acres."**

$\hat{\beta}_1$ is the **price elasticity of supply**: the % change in quantity supplied per 1% change in price.

- For a **10% price increase**: $\Delta\%\ \text{Acres} \approx \hat{\beta}_1 \times 10$

In [ ]:
model2 = smf.ols(
    'np.log(corn_planted_acres) ~ np.log(real_price_lag1) + t',
    data=corn_dta_subset
).fit()
print(model2.summary())

### Reading the Log-Log Output (Acres)

- **`np.log(real_price_lag1)`** (~0.1471): supply elasticity - a 1% increase in lagged real price is associated with a 0.147% increase in acres planted
  - Corn acreage is **price inelastic** (elasticity < 1): farmers cannot easily shift large cropland areas on short notice
  - A **10% increase** in last year's price is associated with roughly a **1.471% increase** in acres planted
- **`t`**: controls for the gradual long-run structural change in planted acreage
- **`R-squared`** (~0.736): the log-log model explains ~74% of the variation in log acreage

### Level-Level vs. Log-Log: Quick Reference

| Feature | Level-Level | Log-Log |
|---|---|---|
| **Dependent variable** | $Y$ in original units | $\ln(Y)$ |
| **Predictor** | $X$ in original units | $\ln(X)$ |
| **Coefficient meaning** | $\Delta Y$ per unit $\Delta X$ | % $\Delta Y$ per 1% $\Delta X$ (elasticity) |
| **Scaling** | Multiply $\hat{\beta}_1$ by the actual price change | Multiply $\hat{\beta}_1$ by the % price change |
| **Corn acres example** | +\$1 price → +1.465M acres; +\$0.10 → +146,500 acres | +1% price → +0.1471% acres; +10% → +1.471% acres |

## Log-Log Model: Corn Production

**Model:** $\ln(\text{Corn Production}) = \beta_0 + \beta_1 \cdot \ln(\text{Real Price}_{t-1}) + \beta_2 \cdot t + \varepsilon$

**Expected sign on $\beta_1$?** The answer depends on the mechanism:

- **Corn acres model:** positive $\beta_1$ expected - farmers respond to higher prices by planting more
- **Corn production model:** positive $\beta_1$ expected - high prices last year should incentivize producers to increase output the following year
- **Hog production model:** negative $\beta_1$ on lagged *corn* price - corn is a major feed input, so higher corn prices raise production costs and reduce hog output

This sign distinction is an exam question.

In [ ]:
model3 = smf.ols(
    'np.log(corn_prod_bu) ~ np.log(real_price_lag1) + t',
    data=corn_dta_subset).fit()
print(model3.summary())

### Reading the Corn Production Log-Log Output

- **`np.log(real_price_lag1)`** (~0.0604): a **1% increase** in lagged real corn price is associated with a **~0.0604% increase** in corn production the following year 
  - A **10% increase** in lagged real corn price is associated with a **~0.604% increase** in corn production the following year
- **`t`**: controls for steady productivity increases over time
- **`R-squared`** (~0.904): **~90%** of the variation in logged corn production is explained by the previous year's logged real prices and the time trend. Most explanatory power comes from the time trend, not the price variable

## R-Squared: Measuring Model Fit

$$R^2 = 1 - \frac{SSR}{SST} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

**Definition:** R-squared is the proportion of variation in $Y$ explained by the model. It ranges from 0 to 1.

| R-squared | Interpretation |
|---|---|
| 0.90 | Model explains 90% of the variation in $Y$ |
| 0.60 | Model explains 60% of the variation |
| 0.03 | Model explains almost none of the variation |

**Important caveat:** R-squared measures statistical fit, not economic correctness. A model can have high R-squared but still produce misleading coefficient estimates compared to our expectations from economic theory. 

In [ ]:
r2_level     = model1.rsquared
r2_loglog_ac = model2.rsquared
r2_loglog_pr = model3.rsquared

print('R-squared comparison across models:')
print(f'  Level-Level acres:  R2 = {r2_level:.3f}  (acres ~ price_lag + trend)')
print(f'  Log-Log acres    :   R2 = {r2_loglog_ac:.3f}  (ln acres ~ ln price_lag + trend)')
print(f'  Log-Log prod     :   R2 = {r2_loglog_pr:.3f}  (ln prod  ~ ln price_lag + trend)')
print()
print('R-squared values for the level-level and log-log models are not directly '
      'comparable. The log-log R-squared reflects fit in log space. Retransforming '
      'fitted values to levels is required for direct comparison across specifications.')

## USDA NASS Data Codes

When downloading raw data from USDA NASS Quick Stats, some `Value` cells contain special codes instead of numbers:

| Code | Meaning |
|---|---|
| `(D)` | Withheld to avoid disclosing data for individual operations |
| `(Z)` | Less than half of the unit shown |

These codes cannot be converted to numbers and must be removed before regression analysis. The following USDA NASS [Quick Stats Glossary](https://quickstats.nass.usda.gov/src/glossary.pdf) contains more information. 

### Handling Non-Numeric Values in Python

```python
value_clean = df['Value'].fillna('').astype(str)

# Remove any row containing (D) or (Z) anywhere
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
df = df[~mask].copy()

# Remove ',' from `Value` column
df['Value'] = (
    df['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# Convert to numeric
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# Check for NAs
print(df['Value'].isna().sum())

# Optional: Drop NAs if needed
# df.dropna(subset=['Value'], inplace=True) 
```

The pre-cleaned CSV files used in this notebook have already had this step applied.

## AR(1) Autoregressive Model

An **AR(1)** model predicts this period's value using only last period's value:

$$P_t = \beta_0 + \beta_1 P_{t-1} + \varepsilon_t$$

The coefficient $\beta_1$ measures **price persistence**:

| $\beta_1$ | Interpretation |
|---|---|
| Near **1.0** | Strong persistence - this year's price closely tracks last year's |
| Near **0** | Weak persistence - last year's price carries little predictive information |

### Worked Example

Given: $P_t = 1.2588 + 0.8710 \cdot P_{t-1}$, and last year's price = \$4.68/bu.

$P_t = 1.2588 + (0.8710 \times 4.68) = 1.2588 + 4.07628 = \$5.34 \text{/bu}$

In [ ]:
# AR(1) calculation: P_t = beta_0 + beta_1 * P_{t-1}
beta_0    = 1.2588
beta_1    = 0.8710
price_lag = 4.68  # last year's price ($/bu)

predicted = beta_0 + beta_1 * price_lag
print(f'P_t = {beta_0} + ({beta_1} x {price_lag}) = ${predicted:.2f}/bu')

In [ ]:
model4 = smf.ols(
    'real_price ~ real_price_lag1',
    data=corn_dta).fit()
print(model4.summary())

## Appendix: Train-Test Split

A **train-test split** evaluates out-of-sample forecast accuracy. For time series, the split must respect temporal order. Future observations cannot be used to train a model that predicts the past.

**Standard approach:** Fit the model on all observations up to a cutoff year; evaluate predictions on the years that follow (i.e., holdout data).

In [ ]:
cutoff_year = 2019  # last training year; 2020-2024 becomes the holdout test set

train = corn_dta[corn_dta['year'] <= cutoff_year].copy()
test  = corn_dta[corn_dta['year'] >  cutoff_year].copy()

tr_min, tr_max = train['year'].min(), train['year'].max()
te_min, te_max = test['year'].min(),  test['year'].max()

print(f'Training set: {tr_min}-{tr_max}  ({len(train)} observations)')
print(f'Test set:     {te_min}-{te_max}   ({len(test)} observations)')
print()
print('Always split in time order. Never use test data during model training.')